# Семинар №12. Контроль качества данных и аналитическая витрина

## Как устроен ноутбук

1. Каждая тема идёт по схеме: **теория → демо → упражнение**.
2. Ячейки с комментарием `# УПРАЖНЕНИЕ N` — для самостоятельной работы; там стоят `TODO`.
3. Выполняйте ячейки **по порядку**: состояние таблиц в БД важно.
4. Сброс: `Kernel → Restart & Run All`.
5. Решений в ноутбуке нет: сначала попробуйте сами, потом сверяйтесь с `scripts/` и разбором на семинаре.

## Окружение

- PostgreSQL 16 слушает `localhost:5432` **внутри контейнера** (trust auth — пароль не нужен).
- База `etl_db` (создаётся автоматически, если её нет).
- Jupyter-пароль: `student`.

## Данные

Файлы с исходными событиями лежат **в одной папке с ноутбуком** (`*.csv`). Ноутбук сам найдёт их, создаст нужные таблицы (`CREATE IF NOT EXISTS` + миграция) и загрузит данные в `events`. Если у вас файлы в подпапке `data/` — код это тоже учтёт.

**Колонки CSV** (по именам в заголовке; порядок может быть любым — код читает заголовок): `event_id, ts, device_id, region, event_type, value, status`.

> `event_id` — **строка** (в нашем датасете это UUID вида `fc39d6a5-51a5-5b4f-9a32-333be8eae051`), поэтому в схеме это `TEXT`. Именно так устроена контрольная сумма в семинаре 12: `md5(event_id::text)`.
>
> `region` и `value` **могут быть NULL** (пустая строка в CSV → NULL через `NULL ''`). Поэтому DQ-проверка «нет NULL в ключевых столбцах» распространяется только на `event_id, ts, event_type, status`.

## План

1. Подготовка окружения: подключение, миграция схемы, загрузка данных, профиль данных.
2. **Тема 2.** Проверка качества данных (DQ-checks).
3. **Тема 3.** Аналитическая витрина и мостик к колоночным СУБД (OLAP + CSV vs Parquet).
4. Итоги.

> Связь с семинаром 11: там мы построили быстрый, но «хрупкий» пайплайн. Здесь — достраиваем контроль качества и аналитический слой поверх готовой витрины `events`.

In [ ]:
# Проверка окружения и общая конфигурация
import sys
print("Python:", sys.version.split()[0])

import psycopg, sqlalchemy, pandas, numpy
print("psycopg     :", psycopg.__version__)
print("sqlalchemy  :", sqlalchemy.__version__)
print("pandas      :", pandas.__version__)
print("numpy       :", numpy.__version__)

try:
    import pyarrow
    print("pyarrow     :", pyarrow.__version__)
except ImportError:
    print("pyarrow     : НЕ УСТАНОВЛЕН — Parquet-упражнение не запустится")

# Параметры подключения. Внутри контейнера localhost = этот же контейнер,
# PG настроен с trust-аутентификацией — пароль НЕ требуется.
DB = {
    "host":   "localhost",
    "port":   5432,
    "dbname": "etl_db",
    "user":   "postgres",
}

PSYCOPG_CONN    = f"host={DB['host']} port={DB['port']} dbname={DB['dbname']} user={DB['user']}"
SQLALCHEMY_URL  = f"postgresql+psycopg://{DB['user']}@{DB['host']}:{DB['port']}/{DB['dbname']}"
PSYCOPG_ADMIN   = f"host={DB['host']} port={DB['port']} dbname=postgres user={DB['user']}"

# База etl_db должна существовать — создаём, если её нет
with psycopg.connect(PSYCOPG_ADMIN, autocommit=True) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT 1 FROM pg_database WHERE datname = %s", (DB["dbname"],))
        if not cur.fetchone():
            cur.execute(f'CREATE DATABASE "{DB["dbname"]}"')
            print(f"\nСоздана БД {DB['dbname']}")
        else:
            print(f"\nБД {DB['dbname']} уже существует")

print("psycopg DSN      :", PSYCOPG_CONN)
print("SQLAlchemy URL   :", SQLALCHEMY_URL)

## Подготовка: миграция схемы и создание таблиц

Три таблицы — как в семинаре 12:

| Таблица | Назначение | Особенности |
|---|---|---|
| `staging_events` | приёмный буфер одного батча | `UNLOGGED`, без ограничений |
| `events` | целевая («золотые» данные) | `PRIMARY KEY (event_id)` — идемпотентность |
| `etl_log` | журнал пайплайна | `kind='batch'` — батчи, `kind='check'` — DQ-проверки |

**Идемпотентность DDL**: `staging_events` пересоздаётся всегда (это скретчпад); `events` и `etl_log` — пересоздаются только если в `events.event_id` **не** `TEXT` (миграция со старой схемы семинара 11/12, где `event_id` мог быть `BIGINT`).

`event_id` в нашем датасете — **UUID-строка**, поэтому это `TEXT`. Прочие колонки намеренно **nullable**: `region`/`value`/`device_id` могут отсутствовать по бизнес-логике; DQ-проверки ловят NULL в **ключевых** столбцах (`event_id, ts, event_type, status`), а не во всех подряд.

In [ ]:
# --- Миграция схемы и создание таблиц ---
# staging_events всегда пересоздаём: это UNLOGGED-скретчпад.
# events и etl_log — только если их схема несовместима
# (в старой версии ноутбука event_id мог быть BIGINT).
with psycopg.connect(PSYCOPG_CONN, autocommit=True) as conn:
    with conn.cursor() as cur:
        cur.execute("DROP TABLE IF EXISTS staging_events CASCADE")

        cur.execute("""
            SELECT data_type
            FROM information_schema.columns
            WHERE table_schema = 'public'
              AND table_name   = 'events'
              AND column_name  = 'event_id'
        """)
        row = cur.fetchone()
        if row and row[0].lower() != "text":
            print(f"⚠ events.event_id = {row[0]}, ожидался text → пересоздаём events и etl_log")
            cur.execute("DROP TABLE IF EXISTS events CASCADE")
            cur.execute("DROP TABLE IF EXISTS etl_log CASCADE")

        cur.execute("""
            CREATE TABLE IF NOT EXISTS events (
                event_id    TEXT PRIMARY KEY,   -- UUID-строка
                ts          TIMESTAMP,
                region      TEXT,
                event_type  TEXT,
                value       NUMERIC,
                status      TEXT,
                device_id   INT
            )
        """)
        cur.execute("""
            CREATE UNLOGGED TABLE IF NOT EXISTS staging_events (
                event_id    TEXT,
                ts          TIMESTAMP,
                region      TEXT,
                event_type  TEXT,
                value       NUMERIC,
                status      TEXT,
                device_id   INT
            )
        """)
        cur.execute("""
            CREATE TABLE IF NOT EXISTS etl_log (
                id            SERIAL PRIMARY KEY,
                kind          TEXT NOT NULL,          -- 'batch' | 'check'
                batch_no      INT,
                rows_loaded   INT,
                rows_expected INT,
                checksum      TEXT,
                check_name    TEXT,
                value         TEXT,
                status        TEXT,
                created_at    TIMESTAMP DEFAULT now()
            )
        """)
        cur.execute("CREATE INDEX IF NOT EXISTS events_region_idx ON events (region)")
        cur.execute("CREATE INDEX IF NOT EXISTS events_ts_idx     ON events (ts)")

print("Схема готова: staging_events, events, etl_log")

### Загрузка данных из файлов

Ноутбук ищет `*.csv` в текущей папке (и в `./data/`, если она есть), исключая файлы экспорта (`*export*`). **Каждый файл трактуется как один батч.**

Загрузка идёт по промышленной схеме: `TRUNCATE staging` → `COPY` → `INSERT ... ON CONFLICT DO NOTHING` → запись в `etl_log`. Именно это даёт идемпотентность (перезапуск ячейки не создаёт дублей).

Обратите внимание:
- **Заголовок CSV читается заранее**, чтобы `COPY` мог подхватить колонки в фактическом порядке (в вашем файле порядок: `event_id, ts, device_id, region, event_type, value, status` — он отличается от порядка колонок в таблице).
- Опция `NULL ''` в `COPY` явно говорит: пустое поле CSV = `NULL`. Так `region` и `value` в первой строке станут `NULL`, а не пустыми строками.

In [ ]:
from pathlib import Path
import csv as _csv

# --- Поиск исходных файлов ---
candidates = sorted(Path(".").glob("*.csv"))
if not candidates and Path("data").is_dir():
    candidates = sorted(Path("data").glob("*.csv"))
source_files = [p for p in candidates if "export" not in p.name.lower()]

print(f"Найдено исходных файлов: {len(source_files)}")
for p in source_files:
    print(f"  {p}  ({p.stat().st_size/1e6:.1f} МБ)")

# --- Проверка: уже загружено? ---
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT count(*) FROM events")
        existing = cur.fetchone()[0]

if existing > 0:
    EXPECTED_TOTAL_ROWS = existing
    with psycopg.connect(PSYCOPG_CONN) as conn:
        with conn.cursor() as cur:
            cur.execute("SELECT count(*) FROM etl_log WHERE kind='batch' AND status='OK'")
            EXPECTED_BATCHES = cur.fetchone()[0]
    print(f"\nevents уже содержит {existing:,} строк — загрузку пропускаем.")
    print(f"Успешных батчей в etl_log: {EXPECTED_BATCHES}")
elif not source_files:
    EXPECTED_TOTAL_ROWS = 0
    EXPECTED_BATCHES = 0
    print("\nИсходные CSV не найдены. Положите файлы в папку с ноутбуком.")
else:
    EXPECTED_TOTAL_ROWS = 0
    EXPECTED_BATCHES = 0
    REQUIRED_COLS = ["event_id", "ts", "region", "event_type", "value", "status", "device_id"]
    for i, path in enumerate(source_files, start=1):
        # 1) Читаем заголовок, чтобы COPY попал в нужные колонки независимо от порядка
        with open(path, "r", encoding="utf-8", newline="") as f:
            header = next(_csv.reader(f))
            header = [c.strip().lstrip("\ufeff").lower() for c in header]
        missing = [c for c in REQUIRED_COLS if c not in header]
        if missing:
            raise ValueError(f"{path.name}: в заголовке нет колонок {missing}. Заголовок: {header}")
        copy_cols = ", ".join(header)   # фактический порядок колонок в CSV

        # 2) Считаем строки данных (без заголовка)
        with open(path, "r", encoding="utf-8") as f:
            n_rows = max(sum(1 for _ in f) - 1, 0)

        # 3) COPY → staging → UPSERT в events → журнал (в одной транзакции)
        with psycopg.connect(PSYCOPG_CONN) as conn:
            with conn.cursor() as cur:
                cur.execute("TRUNCATE staging_events")
                copy_sql = (
                    f"COPY staging_events ({copy_cols}) "
                    "FROM STDIN WITH (FORMAT csv, HEADER true, NULL '')"
                )
                with open(path, "r", encoding="utf-8") as f:
                    with cur.copy(copy_sql) as copy:
                        while True:
                            chunk = f.read(1 << 16)
                            if not chunk:
                                break
                            copy.write(chunk)

                cur.execute("""
                    INSERT INTO events (event_id, ts, region, event_type, value, status, device_id)
                    SELECT event_id, ts, region, event_type, value, status, device_id
                    FROM staging_events
                    ON CONFLICT (event_id) DO NOTHING
                """)
                loaded = cur.rowcount

                cur.execute("""
                    INSERT INTO etl_log (kind, batch_no, rows_loaded, rows_expected, status)
                    VALUES ('batch', %s, %s, %s, 'OK')
                """, (i, loaded, n_rows))
            conn.commit()

        print(f"  батч {i}: {path.name} — загружено {loaded:,} / {n_rows:,}")
        EXPECTED_TOTAL_ROWS += n_rows
        EXPECTED_BATCHES += 1

print(f"\nИтого: строк в events = {EXPECTED_TOTAL_ROWS:,}, батчей = {EXPECTED_BATCHES}")

# --- Сверка ---
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT count(*), count(DISTINCT event_id) FROM events")
        n, n_dist = cur.fetchone()
print(f"Контроль: count(*) = {n:,}, count(DISTINCT event_id) = {n_dist:,}")
assert n == n_dist, "Дубли по event_id — идемпотентность нарушена!"
print("Идемпотентность подтверждена: дублей нет.")

### Демо: профиль данных

Прежде чем писать DQ-проверки, полезно «пощупать» данные: диапазон `ts`, набор `status`, `min/max` для `value` и `device_id`, сколько NULL в каждой колонке. Это поможет осмысленно выбрать пороги в упражнениях (особенно `FRESHNESS_THRESHOLD` и `referential_device_range`).

In [ ]:
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT
                count(*)            AS n,
                min(ts)             AS ts_min,
                max(ts)             AS ts_max,
                min(value)          AS value_min,
                max(value)          AS value_max,
                min(device_id)      AS dev_min,
                max(device_id)      AS dev_max
            FROM events
        """)
        row = cur.fetchone()
        print(f"rows        : {row[0]:,}")
        print(f"ts range    : {row[1]}  ..  {row[2]}")
        print(f"value range : {row[3]}  ..  {row[4]}")
        print(f"device range: {row[5]}  ..  {row[6]}")

        cur.execute("SELECT status, count(*) FROM events GROUP BY status ORDER BY status")
        print("\nstatus distribution:")
        for s, c in cur.fetchall():
            print(f"  {s!r:10}  {c:>10,}")

        print("\nNULL counts:")
        for col in ["event_id", "ts", "region", "event_type", "value", "status", "device_id"]:
            cur.execute(f"SELECT count(*) FROM events WHERE {col} IS NULL")
            print(f"  {col:12s}: {cur.fetchone()[0]:>10,}")

## 2. Проверка качества данных (data quality checks)

Контроль **доставки** (все ли строки доехали, нет ли дублей) мы уже прошли. Теперь — контроль **качества самих данных**: значения в допустимых диапазонах, ключи без пропусков, справочники без опечаток.

### 2.1. Где проверять: БД vs пайплайн

| Механизм | Что гарантирует | Когда ломается |
|---|---|---|
| `CHECK`, `NOT NULL`, `FK` в БД | инварианты **всегда**, для любого пути записи | отвергает строку → массовая загрузка падает целиком |
| DQ-проверки в пайплайне | статистика качества (NULL, диапазоны, полнота) | не блокируют — фиксируют и сигнализируют |

Промышленная практика: **жёсткие** инварианты — в ограничения БД; **мягкие** метрики качества — в DQ-слой пайплайна. Это тот же подход, что в Great Expectations / dbt tests, только без фреймворков: каждая проверка — SQL-запрос + предикат, результат пишется в `etl_log` (`kind='check'`).

### 2.2. Демо: одна проверка и журнал

Ниже — минимальный помощник `run_dq_check`. Он выполняет SQL, применяет предикат к результату, печатает `OK`/`FAIL` и пишет строку в `etl_log`. Покажем на примере проверки полноты (row count).

In [ ]:
# Помощник: выполнить проверку, напечатать статус, записать в etl_log
def run_dq_check(name, sql, predicate, expected_desc=""):
    """
    name          — имя проверки (для журнала)
    sql           — SQL, возвращающий ОДНО скалярное значение
    predicate     — callable(value) -> bool; True = проверка прошла
    expected_desc — что ожидали (для печати)
    Возвращает True/False.
    """
    with psycopg.connect(PSYCOPG_CONN) as conn:
        with conn.cursor() as cur:
            cur.execute(sql)
            value = cur.fetchone()[0]
            ok = bool(predicate(value))
            status = "OK" if ok else "FAIL"
            print(f"{status:5s}  {name:32s}  value={value}  {expected_desc}")
            cur.execute("""
                INSERT INTO etl_log (kind, check_name, value, status)
                VALUES ('check', %s, %s, %s)
            """, (name, str(value), status))
        conn.commit()
    return ok


# Демо: одна проверка — количество строк в events совпадает с ожидаемым
run_dq_check(
    name="row_count_matches_source",
    sql="SELECT count(*) FROM events",
    predicate=lambda v: v == EXPECTED_TOTAL_ROWS,
    expected_desc=f"ожидалось {EXPECTED_TOTAL_ROWS:,}",
)

### Упражнение 1. Набор DQ-проверок для витрины

**Задание.** Реализуйте 4 оставшиеся проверки, покрывающие классические измерения качества данных.

| Проверка | Измерение | SQL-эскиз |
|---|---|---|
| `no_nulls_in_key_columns` | полнота | `SELECT count(*) FROM events WHERE event_id IS NULL OR ts IS NULL OR event_type IS NULL OR status IS NULL` |
| `business_key_unique` | уникальность | `SELECT count(*) FROM (SELECT event_id FROM events GROUP BY event_id HAVING count(*) > 1) t` |
| `value_in_plausible_range` | валидность | `SELECT count(*) FROM events WHERE value IS NOT NULL AND (value < 0 OR value > 1000000)` |
| `referential_device_range` | целостность | `SELECT count(*) FROM events WHERE device_id IS NOT NULL AND (device_id < 1 OR device_id > 100000)` |
| `all_batches_logged` | прослеживаемость | `SELECT count(*) FROM etl_log WHERE kind='batch' AND status='OK'` |

**Про ключевые столбцы.** `region`, `value`, `device_id` в нашем датасете **законно** могут быть NULL (см. профиль выше). Поэтому `no_nulls_in_key_columns` покрывает только `event_id, ts, event_type, status` — без них запись бесполезна для аналитики. Проверять отсутствие NULL в `region`/`value`/`device_id` можно, но это уже не «ключевые столбцы», а отдельные бизнес-правила с другими порогами.

**Шаги.**
1. Для каждой проверки вызовите `run_dq_check(name, sql, predicate, expected_desc)`.
2. Для счётчиков нарушений предикат — `lambda v: v == 0`.
3. Для `all_batches_logged` предикат — `lambda v: v == EXPECTED_BATCHES`.
4. В конце напечатайте итог: `итог: N/M проверок пройдено`.

**Критерий успеха.** Все 5 проверок (считая демо-проверку `row_count_matches_source`) печатают `OK`; в `etl_log` появились строки с `kind='check'`.

> Подсказка: `referential_device_range` — диапазон `device_id`. Сверьтесь с профилем: если у вас `device_id` доходит до 10⁶ — расширьте границу. Смысл проверки — поймать заведомо «мусорные» значения, а не угадать точный максимум.

In [15]:
# УПРАЖНЕНИЕ 1. Набор DQ-проверок

# TODO: проверка полноты — нет NULL в ключевых столбцах
# run_dq_check(
#     name="no_nulls_in_key_columns",
#     sql="SELECT count(*) FROM events WHERE event_id IS NULL OR ts IS NULL OR event_type IS NULL OR status IS NULL",
#     predicate=lambda v: v == 0,
#     expected_desc="ожидалось 0",
# )

# TODO: проверка уникальности бизнес-ключа
# run_dq_check(name="business_key_unique", sql="...", predicate=lambda v: v == 0, expected_desc="ожидалось 0")

# TODO: проверка валидности значения
# run_dq_check(name="value_in_plausible_range", sql="...", predicate=lambda v: v == 0, expected_desc="ожидалось 0")

# TODO: проверка целостности по device_id
# run_dq_check(name="referential_device_range", sql="...", predicate=lambda v: v == 0, expected_desc="ожидалось 0")

# TODO: проверка прослеживаемости — все батчи зафиксированы
# run_dq_check(name="all_batches_logged", sql="...", predicate=lambda v: v == EXPECTED_BATCHES, expected_desc=f"ожидалось {EXPECTED_BATCHES}")

# TODO: напечатать итог: сколько проверок прошло из скольких
# passed = ...; total = ...
# print(f"итог: {passed}/{total} проверок пройдено")

### Упражнение 2. Freshness и согласованность статусов

**Задание.** Добавьте две проверки, которые не сводятся к «счётчик нарушений == 0».

1. **Freshness (свежесть).** Самая свежая запись в витрине не должна быть старше **ожидаемой бизнес-даты** расчёта витрины. Запрос: `SELECT max(ts) FROM events`. Предикат должен сравнивать `max(ts)` с порогом `FRESHNESS_THRESHOLD`, а **не** с `now()`.
   - В комментарии к коду объясните: почему `now()` — плохой порог для исторического датасета?
   - Порог `FRESHNESS_THRESHOLD` задайте сами на основе фактического `max(ts)` из «Профиля данных» выше. Он должен быть **≥ фактического max(ts)**, но не «наивно-сегодня».
2. **Согласованность `status`.** Значения поля `status` — только из справочника `('ok', 'warn', 'crit')`. Запрос: `SELECT count(*) FROM events WHERE status NOT IN ('ok','warn','crit')`. Предикат: `lambda v: v == 0`.

**Дополнительно (для самопроверки).** Искусственно сломайте одну проверку:
- вставьте в `staging_events` одну строку с `status = 'unknown'` (и валидным `event_id`);
- сделайте `INSERT ... ON CONFLICT DO NOTHING` в `events`;
- перезапустите ячейку с проверкой `status_consistency` — она должна напечатать `FAIL`;
- уберите «плохую» строку из `events` (по `event_id`), чтобы не портить витрину для следующих упражнений.

**Критерий успеха.** На «здоровых» данных обе проверки — `OK`; после искусственной порчи — `FAIL`.

In [16]:
# УПРАЖНЕНИЕ 2. Freshness + согласованность status
import datetime as dt

# Порог свежести — ожидаемая бизнес-дата витрины (НЕ now()).
# Поставьте значение >= фактического max(ts) из профиля данных (см. ячейку выше).
FRESHNESS_THRESHOLD = dt.datetime(2024, 3, 31, 23, 59, 59)

# TODO: проверка свежести — max(ts) не старше FRESHNESS_THRESHOLD
# run_dq_check(
#     name="freshness_max_ts",
#     sql="SELECT max(ts) FROM events",
#     predicate=lambda v: v is not None and v <= FRESHNESS_THRESHOLD,
#     expected_desc=f"порог {FRESHNESS_THRESHOLD}",
# )
#
# Комментарий: почему now() — плохой порог для этого датасета?
# ...

# TODO: проверка согласованности статусов
# run_dq_check(
#     name="status_consistency",
#     sql="SELECT count(*) FROM events WHERE status NOT IN ('ok','warn','crit')",
#     predicate=lambda v: v == 0,
#     expected_desc="ожидалось 0",
# )

## 3. Аналитическая витрина и мостик к колоночным СУБД

Данные доставлены, проверки качества пройдены — витрину не стыдно отдать аналитикам. Последний шаг: посмотреть на неё глазами OLAP-запросов и на живом примере увидеть, почему лекция СУБД_7 настаивает на **колоночном хранении** для аналитики.

### 3.1. Демо: OLAP-запрос и `EXPLAIN`

Типичные OLAP-запросы к витрине:
- агрегация по измерению (`region`);
- динамика по дням и типам событий.

Посмотрим на `EXPLAIN (ANALYZE, BUFFERS)` первого запроса. Вывод покажет, что PostgreSQL (NSM-модель хранения) физически читает **все 7 столбцов** таблицы, хотя запросу нужны 2. Это не «плохой план», а следствие физической модели хранения.

In [ ]:
from sqlalchemy import create_engine, text
engine = create_engine(SQLALCHEMY_URL, echo=False)

# Демо: агрегация по регионам
with engine.connect() as conn:
    rows = conn.execute(text("""
        SELECT region, avg(value) AS avg_value, count(*) AS n
        FROM events
        GROUP BY region
        ORDER BY avg_value DESC NULLS LAST
    """)).fetchall()

print(f"{len(rows)} регионов:")
for r in rows:
    print(f"  {str(r[0]):20s}  avg={r[1]}  n={r[2]:>10,}")

# Демо: план выполнения того же запроса
with engine.connect() as conn:
    plan = conn.execute(text("""
        EXPLAIN (ANALYZE, BUFFERS)
        SELECT region, avg(value) AS avg_value, count(*) AS n
        FROM events
        GROUP BY region
        ORDER BY avg_value DESC NULLS LAST
    """)).fetchall()

print("\n--- EXPLAIN (ANALYZE, BUFFERS) ---")
for line in plan:
    print(line[0])

### Упражнение 3. OLAP-запросы к витрине

**Задание.** Напишите три OLAP-запроса и проанализируйте их планы.

**Шаги.**
1. **Агрегация по регионам.** Для каждого `region` посчитайте `avg(value)` и `count(*)`, отсортируйте по среднему значению. Выведите **топ-10**. Учтите NULL в `region` (например, `WHERE region IS NOT NULL` или `ORDER BY ... NULLS LAST`).
2. **Динамика по дням.** Для каждой пары `(date_trunc('day', ts), event_type)` посчитайте `count(*)`. Отсортируйте по дате, затем по типу.
3. **План.** Выполните `EXPLAIN (ANALYZE, BUFFERS)` для запроса из шага 1. Найдите в выводе:
   - тип сканирования (`Seq Scan` / `Index Scan` / `Parallel Seq Scan`);
   - число прочитанных буферов;
   - время выполнения.
4. **Вопрос на подумать** (ответьте в комментарии): почему PostgreSQL читает все 7 столбцов, хотя запросу нужны только `region` и `value`? Как это связано с NSM-моделью хранения из лекции СУБД_7?

**Критерий успеха.** Три запроса выполняются без ошибок; в плане видно `Seq Scan` (или `Parallel Seq Scan`); в комментарии — 1–2 предложения про NSM.

In [18]:
# УПРАЖНЕНИЕ 3. OLAP-запросы

# TODO 1: топ-10 регионов по avg(value) с count(*)
# with engine.connect() as conn:
#     rows = conn.execute(text("""
#         SELECT ...
#         FROM events
#         GROUP BY region
#         ORDER BY ...
#         LIMIT 10
#     """)).fetchall()
# for r in rows: print(r)

# TODO 2: динамика по дням и типам событий
# with engine.connect() as conn:
#     rows = conn.execute(text("""
#         SELECT date_trunc('day', ts) AS day, event_type, count(*) AS n
#         FROM events
#         GROUP BY day, event_type
#         ORDER BY day, event_type
#     """)).fetchall()
# for r in rows[:10]: print(r)

# TODO 3: план EXPLAIN (ANALYZE, BUFFERS) для запроса из TODO 1
# with engine.connect() as conn:
#     plan = conn.execute(text("""
#         EXPLAIN (ANALYZE, BUFFERS)
#         SELECT ...
#     """)).fetchall()
# for line in plan: print(line[0])

# Комментарий: почему читаются все 7 столбцов? Связь с NSM.
# ...

### 3.2. Демо: CSV vs Parquet — DSM/PAX на ладони

Выгрузим `events` в два файла и сравним размер + скорость чтения **двух** столбцов из семи. На живых числах увидим, почему data lake-стеки хранят данные в Parquet, а аналитические СУБД — в собственных колоночных форматах.

Ключевые идеи (прямая иллюстрация лекции СУБД_7):
- Parquet — PAX-подобный формат: внутри row group данные лежат **поколоночно**, поэтому читаются только нужные колонки → меньше I/O (DSM-принцип).
- Однотипные значения в колонке жмутся специализированными кодеками: **dictionary** (повторяющиеся строки → короткие коды), **RLE** (серии одинаковых значений), сверху — snappy.
- CSV — построчный текст: чтобы получить 2 столбца, надо распарсить все 7.

In [20]:
pip install pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 25.6 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import time

# Читаем всю витрину в DataFrame (один раз — дальше работаем с ним)
df = pd.read_sql("SELECT * FROM events", engine)
print(f"DataFrame: {len(df):,} строк, {df.shape[1]} столбцов")

csv_path     = "events_export.csv"
parquet_path = "events_export.parquet"

t0 = time.perf_counter()
df.to_csv(csv_path, index=False)
t_csv_write = time.perf_counter() - t0

t0 = time.perf_counter()
df.to_parquet(parquet_path, compression="snappy", index=False)
t_parq_write = time.perf_counter() - t0

import os
size_csv = os.path.getsize(csv_path) / 1e6
size_prq = os.path.getsize(parquet_path) / 1e6

print(f"\nCSV     : {size_csv:6.1f} МБ, запись {t_csv_write:.2f} c")
print(f"Parquet : {size_prq:6.1f} МБ, запись {t_parq_write:.2f} c")
print(f"Parquet меньше CSV в {size_csv/size_prq:.1f}x")

# Чтение 2 столбцов + groupby
t0 = time.perf_counter()
tmp1 = pd.read_csv(csv_path, usecols=["region", "value"])
agg1 = tmp1.groupby("region")["value"].mean()
t_csv_read = time.perf_counter() - t0

t0 = time.perf_counter()
tmp2 = pd.read_parquet(parquet_path, columns=["region", "value"])
agg2 = tmp2.groupby("region")["value"].mean()
t_prq_read = time.perf_counter() - t0

print(f"\nЧтение 2 столбцов + groupby:")
print(f"  CSV     : {t_csv_read:.2f} c")
print(f"  Parquet : {t_prq_read:.2f} c  → быстрее в {t_csv_read/max(t_prq_read,1e-9):.1f}x")

# Контроль: агрегаты должны совпасть
assert agg1.round(6).equals(agg2.round(6)), "Агрегаты не совпали!"
print("Агрегаты CSV и Parquet идентичны — данные не потеряны.")

### Упражнение 4. CSV vs Parquet: свои замеры и вывод

**Задание.** Повторите замеры на **своих** данных и сформулируйте вывод.

**Шаги.**
1. Убедитесь, что `events_export.csv` и `events_export.parquet` уже созданы демо-ячейкой (или создайте их заново).
2. Соберите **сводную таблицу** с метриками:
   - размер файла (МБ);
   - время записи (с);
   - время чтения 2 столбцов (`region`, `value`) + groupby (с);
   - ускорение/сжатие относительно CSV.
3. **Дополнительно.** Поэкспериментируйте со сжатием Parquet: сравните `compression="snappy"` и `compression="gzip"` (или `None`). Какое даёт меньший файл? Какое быстрее пишется?
4. **Вопрос на подумать** (ответьте в комментарии): почему Parquet быстрее читает 2 столбца, хотя «данных столько же»? Свяжите с **PAX**, **dictionary-кодированием** и **I/O**.

**Критерий успеха.** В выводе — таблица CSV/Parquet с 4 метриками; в комментарии — 2–3 предложения про PAX и dictionary-кодирование.

> **Важно:** тайминги зависят от машины. Контролируйте **соотношение** CSV/Parquet, а не абсолютные секунды. Размер файлов воспроизводится точно, потому что датасет детерминирован.

In [ ]:
# УПРАЖНЕНИЕ 4. Свои замеры CSV vs Parquet
import os, time

# TODO 1: убедиться, что файлы существуют (или создать заново через df.to_csv / df.to_parquet)

# TODO 2: сводная таблица метрик
# results = []
# for label, write_fn, read_fn in [...]:
#     t0 = time.perf_counter(); write_fn(); t_w = time.perf_counter() - t0
#     t0 = time.perf_counter(); read_fn();  t_r = time.perf_counter() - t0
#     size = ...
#     results.append({"format": label, "size_mb": size, "write_s": t_w, "read_s": t_r})
# print(pd.DataFrame(results).to_string(index=False))

# TODO 3 (доп.): сравнение compression='snappy' vs 'gzip' vs None
# for comp in [None, "snappy", "gzip"]:
#     p = f"events_export_{comp or 'none'}.parquet"
#     t0 = time.perf_counter()
#     df.to_parquet(p, compression=comp, index=False)
#     print(f"{str(comp):8s}  size={os.path.getsize(p)/1e6:6.1f} МБ  write={time.perf_counter()-t0:.2f}c")

# Комментарий: почему Parquet быстрее читает 2 столбца (PAX, dictionary, I/O)
# ...

## 4. Итоги

- Промышленный пайплайн = скорость (семинар 11) + **контроль качества**: DQ-проверки, журнал, сверка.
- **DQ-проверки** — мягкий слой качества поверх жёстких ограничений БД. Они не блокируют загрузку, но фиксируют проблемы в `etl_log` и сигнализируют (вплоть до exit code 1 в CI).
- **Свежесть** данных сравнивают с **ожидаемой бизнес-датой**, а не с `now()` — датасеты бывают историческими.
- **OLAP vs OLTP.** OLAP-запросы читают мало столбцов из многих строк — ровно тот паттерн, под который заточены колоночные СУБД и Parquet.
- **Parquet** на практике показал, что колоночное хранение + сжатие дают и меньший размер, и меньший I/O на чтении подмножества столбцов.

**Вопросы на выход:**
1. Чем DQ-проверка отличается от `CHECK`-ограничения? Почему нужны оба?
2. Почему сравнивать `max(ts)` с `now()` для исторического датасета — ошибка?
3. Почему чтение двух столбцов из Parquet быстрее, чем из CSV, хотя данных «столько же»?

## Очистка окружения (опционально)

Файлы `events_export.csv` / `events_export.parquet` и БД `etl_db` можно удалить вручную — ноутбук пересоздаст всё заново при следующем запуске (`Kernel → Restart & Run All`).